# Step 3 · Exploratory Data Analysis (EDA)

**Input:** `data/processed/wq_curated.parquet` from Step 2. Only **analysis-ready** rows are used: single readings in standard units, with no QC, rejected, duplicate, censored, impossible or suspect values.
**Goal:** understand the water-quality data *before* building the warehouse and models, and give first, evidence-based answers to the **mandatory business questions Q1–Q8**.

| Brief requirement (§5) | Section |
|---|---|
| A. Numerical analysis (mean, median, min, max, variance, std, quartiles, IQR, skewness, kurtosis, histograms, box plots) | 1 |
| B. Categorical analysis (frequency, % contribution, rare categories, cross-tabs with outcomes) | 2 |
| C. Temporal analysis (yearly, quarterly, monthly, weekly, day-of-week, time band, peak/low periods) | 3 |
| D. Geographic / entity analysis (regions, states, sites, organizations, map) | 4 |
| E. Outcome analysis (**counts vs rates**, no causal claims) | 5 |
| Q5: relationships among pH, DO, turbidity, nutrients (co-located data) | 6 |
| Answers to Q1–Q8 | 7 |

Every chart has *why this chart* and an **🔎 Inference** written from the actual output.
**Reading rule:** a *count* says how much was measured; a *rate* says how often it was bad. High volume is not the same as high risk.

## 0 · Setup
DuckDB reads the curated Parquet file directly; a view `a` holds only analysis-ready rows.

In [ ]:
import duckdb, pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns, json
from pathlib import Path
pd.set_option('display.max_rows', 80); pd.set_option('display.width', 180)
sns.set_theme(style='whitegrid', context='notebook')
ROOT = Path.cwd().resolve()
for p in [ROOT, *ROOT.parents]:
    if (p / 'data' / 'processed' / 'wq_curated.parquet').exists(): ROOT = p; break
CUR = ROOT / 'data' / 'processed' / 'wq_curated.parquet'
OUT = ROOT / '02_eda'; FIG, TAB = OUT / 'figures', OUT / 'tables'
FIG.mkdir(parents=True, exist_ok=True); TAB.mkdir(parents=True, exist_ok=True)
con = duckdb.connect(); con.execute("SET memory_limit='3GB'")
con.execute(f"CREATE VIEW cur AS SELECT * FROM '{CUR.as_posix()}'")
con.execute("CREATE VIEW a AS SELECT * FROM cur WHERE analysis_ready")
def q(sql): return con.execute(sql).df()
def save_tab(df, name, index=False): df.to_csv(TAB / f'E{name}.csv', index=index); return df
def save_fig(name): plt.tight_layout(); plt.savefig(FIG / f'E{name}.png', dpi=130, bbox_inches='tight'); plt.show()
PARAMS = ['Temperature, water', 'Dissolved oxygen (DO)', 'pH', 'Turbidity', 'Conductivity', 'Nitrate']
SHORT = {'Temperature, water': 'Temperature', 'Dissolved oxygen (DO)': 'Dissolved oxygen', 'pH': 'pH', 'Turbidity': 'Turbidity', 'Conductivity': 'Conductivity', 'Nitrate': 'Nitrate'}
UNIT = {'Temperature, water': '°C', 'Dissolved oxygen (DO)': 'mg/L', 'pH': 'pH', 'Turbidity': 'NTU', 'Conductivity': 'µS/cm', 'Nitrate': 'mg/L as N'}
LOGP = {'Turbidity', 'Conductivity', 'Nitrate'}
ov = q('''SELECT count(*) AS results, count(DISTINCT site) AS sites, count(DISTINCT org) AS organizations, count(DISTINCT state) AS states,
                 min(dt) AS first_date, max(dt) AS last_date, sum(exceedance_flag) AS exceedances, sum(high_risk_flag) AS high_risk FROM a''').T
display(save_tab(ov.rename(columns={0: 'Value'}), '00_overview', index=True))

## 1 · Numerical analysis (brief §5A)
**What:** the full set of summary statistics for each parameter, followed by histograms and box plots.
**Why:** this describes the *typical* value, the spread and the shape of each parameter after cleaning. Skewed parameters need medians, not means, and log scales.

In [ ]:
rows = []
for p in PARAMS:
    r = q(f'''SELECT count(*) n, avg(value_std) mean, median(value_std) median, min(value_std) min, max(value_std) max,
                     var_samp(value_std) variance, stddev(value_std) std, quantile_cont(value_std, 0.25) q1, quantile_cont(value_std, 0.75) q3,
                     skewness(value_std) skewness, kurtosis(value_std) kurtosis FROM a WHERE param = '{p}' ''').iloc[0]
    rows.append({'Parameter': SHORT[p], 'Unit': UNIT[p], **r.to_dict()})
num = pd.DataFrame(rows).set_index('Parameter'); num['IQR'] = num.q3 - num.q1
num['CV (std/mean)'] = num['std'] / num['mean']
display(save_tab(num.round(3), '01_numeric_summary', index=True))
samp = q("SELECT param, value_std FROM (SELECT param, value_std FROM a) USING SAMPLE reservoir(600000 ROWS) REPEATABLE (3)")
fig, axes = plt.subplots(2, 6, figsize=(20, 7), gridspec_kw={'height_ratios': [2.2, 1]})
for i, p in enumerate(PARAMS):
    s = samp[samp.param == p].value_std; ax, bx = axes[0, i], axes[1, i]
    if p in LOGP:
        s = s[s > 0]; bins = np.logspace(np.log10(s.quantile(0.001)), np.log10(s.quantile(0.999)), 50)
        ax.hist(s, bins=bins, color='#4C72B0'); ax.set_xscale('log'); bx.boxplot(s, vert=False, whis=3, flierprops=dict(marker='.', markersize=1, alpha=0.2)); bx.set_xscale('log')
    else:
        ax.hist(s, bins=50, color='#4C72B0'); bx.boxplot(s, vert=False, whis=3, flierprops=dict(marker='.', markersize=1, alpha=0.2))
    ax.axvline(num.loc[SHORT[p], 'median'], color='red', ls='--'); ax.axvline(num.loc[SHORT[p], 'mean'], color='orange', ls=':')
    ax.set(title=f"{SHORT[p]} ({UNIT[p]})\nmedian {num.loc[SHORT[p], 'median']:.2f} · mean {num.loc[SHORT[p], 'mean']:.2f}", ylabel='count' if i == 0 else '')
    bx.set(yticks=[], xlabel=UNIT[p])
fig.suptitle('E1 · Distributions after cleaning: histogram (red = median, orange = mean) and box plot (3×IQR whiskers)', y=1.01, fontsize=13)
save_fig('01_distributions')

**Why this chart:** the histogram shows the shape, and the box plot under it shows the median, the middle 50% and the extremes on the same axis. The red median vs orange mean lines make skew visible: the further apart they are, the more lopsided the data.

**🔎 Inference (E1):** _written after the run_

## 2 · Categorical analysis (brief §5B)
**What:** frequency and % contribution of the main categories, **rare categories** (< 1% share), and cross-tabulations with the outcome (**exceedance rate**).
**Why:** categories become the warehouse *dimensions* and the model *features*. Rare categories may need grouping, and a cross-tab shows which categories go together with poor water quality.

In [ ]:
CATS = {'Region': 'region', 'Water type': 'water_type', 'Sample type': 'sample_type_group', 'Season': 'season',
        'Time band': 'time_band', 'Parameter group': 'parameter_group', 'Equipment group': 'equipment_group', 'Result status': 'status'}
freq_all, rare_all = [], []
for name, col in CATS.items():
    d = q(f"SELECT coalesce({col}, '(blank)') AS category, count(*) n, avg(exceedance_flag) * 100 AS exceed_rate FROM a GROUP BY 1 ORDER BY n DESC")
    d['pct'] = d.n / d.n.sum() * 100; d.insert(0, 'Attribute', name); freq_all.append(d)
    rare_all.append(d[d.pct < 1])
freq = pd.concat(freq_all); rare = pd.concat(rare_all)
display(save_tab(freq.round(2), '02_category_frequency'))
print('Rare categories (< 1% of results):'); display(save_tab(rare.round(3), '02_rare_categories'))

fig, axes = plt.subplots(2, 4, figsize=(20, 8))
for ax, (name, col) in zip(axes.flat, CATS.items()):
    d = freq[freq.Attribute == name].head(8).iloc[::-1]
    bars = ax.barh(d.category.astype(str).str[:28], d.pct, color='#4C72B0')
    for b_, (p_, e_) in zip(bars, zip(d.pct, d.exceed_rate)):
        ax.text(b_.get_width(), b_.get_y() + b_.get_height() / 2, f' {p_:.1f}% · exc {e_:.1f}%', va='center', fontsize=7.5)
    ax.set(title=name, xlabel='% of results'); ax.set_xlim(right=d.pct.max() * 1.6)
fig.suptitle('E2a · Category share (bar) and exceedance rate inside each category (label "exc")', y=1.01, fontsize=13)
save_fig('02_categories')

ct = q('''SELECT region, param, avg(exceedance_flag) * 100 AS rate, count(*) n FROM a WHERE region <> 'Unknown' GROUP BY ALL''')
ctp = ct.pivot(index='region', columns='param', values='rate')[PARAMS].rename(columns=SHORT)
ctn = ct.pivot(index='region', columns='param', values='n')[PARAMS].rename(columns=SHORT)
save_tab(ctp.round(2), '02_crosstab_region_param_exceed', index=True)
wt = q('''SELECT water_type, param, avg(exceedance_flag) * 100 AS rate, count(*) n FROM a
          WHERE water_type IN ('Surface Water', 'Groundwater', 'Ocean Water', 'Not Reported') GROUP BY ALL''')
wtp = wt.pivot(index='water_type', columns='param', values='rate')[PARAMS].rename(columns=SHORT)
save_tab(wtp.round(2), '02_crosstab_watertype_param_exceed', index=True)
fig, axes = plt.subplots(1, 2, figsize=(17, 4.6))
sns.heatmap(ctp, annot=True, fmt='.1f', cmap='OrRd', ax=axes[0], cbar_kws={'label': 'exceedance %'})
axes[0].set(title='E2b · Exceedance rate (%): region × parameter', xlabel='', ylabel='')
sns.heatmap(wtp, annot=True, fmt='.1f', cmap='OrRd', ax=axes[1], cbar_kws={'label': 'exceedance %'})
axes[1].set(title='E2c · Exceedance rate (%): water type × parameter', xlabel='', ylabel='')
save_fig('02_crosstabs')

**Why these charts:** the bars show how big each category is, *and* (in the label) how often its results exceed thresholds. The heat maps cross two categories with the outcome rate, so hot cells show the combinations with the most problems.

**🔎 Inference (E2):** _written after the run_

## 3 · Temporal analysis (brief §5C) · answers **Q1** (measurements over time) and **Q4** (seasonal variation)
**What:** volume and values by year, quarter, month, week, day of week and time band, then seasonal patterns per parameter.
**Why:** sampling effort changes over the year, so we look at both *how much was measured* and *what the water was like* (medians and exceedance rates).

In [ ]:
yr = q('SELECT year, count(*) results, count(DISTINCT site) sites, count(DISTINCT org) orgs, avg(exceedance_flag) * 100 exceed_rate FROM a GROUP BY 1 ORDER BY 1')
qt = q("SELECT year || '-Q' || quarter AS quarter, count(*) results, avg(exceedance_flag) * 100 exceed_rate FROM a GROUP BY 1 ORDER BY 1")
mo = q("SELECT strftime(dt, '%Y-%m') AS month, count(*) results, count(DISTINCT site) sites, avg(exceedance_flag) * 100 exceed_rate FROM a GROUP BY 1 ORDER BY 1")
wk = q("SELECT date_trunc('week', dt) AS week, count(*) results FROM a GROUP BY 1 ORDER BY 1")
dw = q("SELECT dayofweek(dt) AS d, day_of_week, count(*) results, avg(exceedance_flag) * 100 exceed_rate FROM a GROUP BY ALL ORDER BY d")
tbd = q("SELECT time_band, count(*) results, avg(exceedance_flag) * 100 exceed_rate FROM a GROUP BY 1 ORDER BY results DESC")
for n_, d in [('03_yearly', yr), ('03_quarterly', qt), ('03_monthly', mo), ('03_weekly', wk), ('03_day_of_week', dw), ('03_time_band', tbd)]: save_tab(d.round(3), n_)
display(yr.round(2)); display(qt.round(2))
peak, low = mo.loc[mo.results.idxmax()], mo.loc[mo.results.idxmin()]
print(f'Peak month: {peak.month} ({peak.results:,} results) · lowest month: {low.month} ({low.results:,} results)')

fig, axes = plt.subplots(2, 2, figsize=(17, 8))
ax = axes[0, 0]; ax.bar(mo.month, mo.results / 1e3, color=['#4C72B0' if m < '2025' else '#DD8452' for m in mo.month])
ax2 = ax.twinx(); ax2.plot(mo.month, mo.sites, color='black', marker='.', label='active sites'); ax2.set_ylabel('active sites'); ax2.grid(False)
ax.set(title='E3a · Results per month (bars) and active sites (line)', ylabel='results (thousands)'); ax.tick_params(axis='x', rotation=75, labelsize=7.5)
axes[0, 1].plot(wk.week, wk.results / 1e3, color='#4C72B0'); axes[0, 1].set(title='E3b · Results per week', ylabel='results (thousands)')
order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
dw2 = dw.set_index('day_of_week').loc[order]
axes[1, 0].bar(dw2.index, dw2.results / 1e3, color='#55A868'); a3 = axes[1, 0].twinx(); a3.plot(dw2.index, dw2.exceed_rate, color='#C44E52', marker='o'); a3.grid(False)
a3.set_ylabel('exceedance %', color='#C44E52'); axes[1, 0].set(title='E3c · Day of week: volume (bars) and exceedance rate (red)', ylabel='results (thousands)')
tb2 = tbd.set_index('time_band').reindex(['Night', 'Morning', 'Afternoon', 'Evening', 'Whole day', 'Not Reported']).dropna()
axes[1, 1].bar(tb2.index, tb2.results / 1e3, color='#8172B2'); a4 = axes[1, 1].twinx(); a4.plot(tb2.index, tb2.exceed_rate, color='#C44E52', marker='o'); a4.grid(False)
a4.set_ylabel('exceedance %', color='#C44E52'); axes[1, 1].set(title='E3d · Time band: volume and exceedance rate', ylabel='results (thousands)')
save_fig('03_temporal_volume')

mp = q("SELECT param, strftime(dt, '%Y-%m') AS month, median(value_std) med, count(*) n FROM a GROUP BY ALL ORDER BY month")
save_tab(mp.round(3), '03_monthly_parameter_median')
fig, axes = plt.subplots(2, 3, figsize=(17, 7.5))
for ax, p in zip(axes.flat, PARAMS):
    d = mp[mp.param == p]; d = d.assign(yr=d.month.str[:4], m=d.month.str[5:].astype(int))
    for y, c in [('2024', '#4C72B0'), ('2025', '#DD8452')]:
        dd = d[d.yr == y]; ax.plot(dd.m, dd.med, marker='o', color=c, label=y)
    ax.set(title=f'{SHORT[p]} – monthly median ({UNIT[p]})', xticks=range(1, 13), xlabel='month'); ax.legend(fontsize=8)
fig.suptitle('E3e · Q1/Q4: monthly median of each parameter, 2024 vs 2025', y=1.01, fontsize=13)
save_fig('03_monthly_parameter_trend')

ss = q("SELECT param, season, median(value_std) med, avg(exceedance_flag) * 100 exc, count(*) n FROM a GROUP BY ALL")
so = ['Winter', 'Spring', 'Summer', 'Autumn']
sm = ss.pivot(index='param', columns='season', values='med').loc[PARAMS, so].rename(index=SHORT)
se = ss.pivot(index='param', columns='season', values='exc').loc[PARAMS, so].rename(index=SHORT)
save_tab(sm.round(3), '03_seasonal_median', index=True); save_tab(se.round(2), '03_seasonal_exceedance', index=True)
sbox = q("SELECT param, season, value_std FROM (SELECT param, season, value_std FROM a) USING SAMPLE reservoir(400000 ROWS) REPEATABLE (5)")
fig, axes = plt.subplots(1, 7, figsize=(22, 4.5), gridspec_kw={'width_ratios': [1] * 6 + [1.6]})
for ax, p in zip(axes[:6], PARAMS):
    d = sbox[sbox.param == p]
    if p in LOGP: d = d[d.value_std > 0]; ax.set_yscale('log')
    sns.boxplot(data=d, x='season', y='value_std', order=so, ax=ax, showfliers=False, palette='coolwarm', hue='season', legend=False)
    ax.set(title=SHORT[p], xlabel='', ylabel=UNIT[p]); ax.tick_params(axis='x', rotation=45)
sns.heatmap(se, annot=True, fmt='.1f', cmap='OrRd', ax=axes[6], cbar_kws={'label': 'exceedance %'}); axes[6].set(title='Exceedance % by season', xlabel='', ylabel='')
fig.suptitle('E3f · Q4: seasonal variation (box plots without outliers) and seasonal exceedance rate', y=1.03, fontsize=13)
save_fig('03_seasonal')

**Why these charts:** monthly bars with an active-site line separate *more sampling* from *more sites*. Weekly and day-of-week plots show the sampling rhythm, and the red lines put the outcome *rate* next to the volume. The 2024-vs-2025 lines test whether seasonal patterns repeat, and the seasonal box plots and heat map show the actual water-quality change by season.

**🔎 Inference (E3):** _written after the run_

## 4 · Geographic and entity analysis (brief §5D) · answers **Q2** (compare sites), **Q3** (worst sites), **Q7** (coverage), **Q8** (location types)
**Fairness rule:** a site needs **at least 30 analysis-ready results for a parameter** before it is ranked. Without this, a site with 2 samples could top a "worst sites" list by chance.

In [ ]:
MINN = 30
sm_ = q(f'''SELECT site, param, any_value(state) state, any_value(region) region, any_value(org) org, count(*) n, median(value_std) med,
                   avg(exceedance_flag) * 100 exc_rate, sum(exceedance_flag) exc_n, avg(high_risk_flag) * 100 hr_rate
            FROM a GROUP BY site, param HAVING count(*) >= {MINN}''')
save_tab(sm_.round(3), '04_site_parameter_stats')
spread = sm_.groupby('param').med.describe(percentiles=[0.1, 0.5, 0.9])[['count', '10%', '50%', '90%', 'min', 'max']].loc[PARAMS].rename(index=SHORT)
spread.columns = ['sites (≥30 results)', 'P10 of site medians', 'median of site medians', 'P90 of site medians', 'lowest site median', 'highest site median']
display(save_tab(spread.round(3), '04_Q2_site_median_spread', index=True))
worst = (sm_.sort_values(['param', 'exc_rate', 'n'], ascending=[True, False, False]).groupby('param').head(5))
worst = worst[worst.exc_rate > 0][['param', 'site', 'state', 'org', 'n', 'med', 'exc_rate', 'hr_rate']]
worst['param'] = worst.param.map(SHORT); display(save_tab(worst.round(2), '04_Q3_worst_sites'))
poor = sm_.assign(poor=sm_.exc_rate >= 50).groupby('param').agg(sites=('site', 'count'), poor_sites=('poor', 'sum')).loc[PARAMS].rename(index=SHORT)
poor['% poor (≥50% exceed)'] = (poor.poor_sites / poor.sites * 100).round(1); display(save_tab(poor, '04_Q3_poor_sites', index=True))

stt = q('''SELECT state, region, count(*) results, count(DISTINCT site) sites, count(DISTINCT org) orgs, avg(exceedance_flag) * 100 exc_rate,
                  sum(exceedance_flag) exc_n FROM a WHERE state NOT IN ('Unknown') GROUP BY ALL ORDER BY results DESC''')
save_tab(stt.round(2), '04_state_summary')
orgc = q('''SELECT org, count(*) results, count(DISTINCT site) sites, count(DISTINCT param) params, count(DISTINCT strftime(dt, '%Y-%m')) months,
                   count(DISTINCT state) states, avg(exceedance_flag) * 100 exc_rate FROM a GROUP BY org ORDER BY results DESC''')
display(save_tab(orgc.head(15).round(2), '04_Q7_org_coverage'))
covq = pd.DataFrame({'Value': {'Organizations': len(orgc), 'Orgs covering all 24 months': int((orgc.months == 24).sum()),
       'Orgs measuring all 6 parameters': int((orgc.params == 6).sum()), 'Median sites per org': orgc.sites.median(),
       'Share of results from top 10 orgs (%)': round(orgc.results.head(10).sum() / orgc.results.sum() * 100, 1)}})
display(save_tab(covq, '04_Q7_coverage_summary', index=True))
wtq = q('''SELECT water_type, param, count(*) n, median(value_std) med, avg(exceedance_flag) * 100 exc FROM a
           WHERE water_type IN ('Surface Water', 'Groundwater', 'Ocean Water') GROUP BY ALL''')
wtm = wtq.pivot(index='water_type', columns='param', values='med')[PARAMS].rename(columns=SHORT)
wtn = wtq.pivot(index='water_type', columns='param', values='n')[PARAMS].rename(columns=SHORT)
display(save_tab(wtm.round(2), '04_Q8_watertype_median', index=True)); save_tab(wtn, '04_Q8_watertype_counts', index=True)

sites_map = q('''SELECT site, median(lat) lat, median(lon) lon, count(*) n, avg(exceedance_flag) * 100 exc FROM a
                 WHERE lat IS NOT NULL GROUP BY site HAVING count(*) >= 30''')
gj = json.load(open(ROOT / 'data' / 'reference' / 'us-states.json'))
fig, axes = plt.subplots(1, 2, figsize=(19, 6), gridspec_kw={'width_ratios': [1.45, 1]})
for f in gj['features']:
    g = f['geometry']; rings = [g['coordinates']] if g['type'] == 'Polygon' else g['coordinates']
    for poly in rings: r_ = np.array(poly[0]); axes[0].plot(r_[:, 0], r_[:, 1], color='grey', lw=0.3)
sc = axes[0].scatter(sites_map.lon, sites_map.lat, c=sites_map.exc.clip(upper=60), s=np.clip(np.sqrt(sites_map.n) / 3, 2, 30), cmap='RdYlGn_r', alpha=0.7)
plt.colorbar(sc, ax=axes[0], label='site exceedance rate % (capped at 60)')
axes[0].set(xlim=(-170, -64), ylim=(17, 72), title=f'E4a · Sites with ≥30 results ({len(sites_map):,}): colour = exceedance rate, size = volume')
t15 = stt.head(15)
axes[1].scatter(t15.results / 1e3, t15.exc_rate, s=t15.sites / 5 + 20, color='#4C72B0', alpha=0.7)
for _, r_ in t15.iterrows(): axes[1].annotate(r_.state, (r_.results / 1e3, r_.exc_rate), fontsize=8, xytext=(4, 2), textcoords='offset points')
axes[1].set(title='E4b · Top 15 states: volume vs exceedance rate (bubble = sites)', xlabel='results (thousands)', ylabel='exceedance rate %')
save_fig('04_geography')

top_sites = q('SELECT site FROM a GROUP BY site ORDER BY count(*) DESC LIMIT 40').site
hm = q(f'''SELECT site, strftime(dt, '%Y-%m') m, count(*) n FROM a WHERE site IN ({', '.join(repr(s) for s in top_sites)}) GROUP BY ALL''')
hmp = hm.pivot(index='site', columns='m', values='n').reindex(top_sites).fillna(0)
fig, axes = plt.subplots(1, 2, figsize=(20, 8), gridspec_kw={'width_ratios': [1.6, 1]})
sns.heatmap(np.log10(hmp + 1), cmap='viridis', ax=axes[0], cbar_kws={'label': 'log10(results + 1)'})
axes[0].set(title='E4c · Q7: Site × month coverage (40 busiest sites)', xlabel='', ylabel=''); axes[0].tick_params(axis='y', labelsize=6.5)
wtm_n = (wtm / wtm.loc['Surface Water'])
sns.heatmap(wtm, annot=True, fmt='.2f', cmap='Blues', ax=axes[1], cbar=False)
axes[1].set(title='E4d · Q8: median value by water type (location type proxy)', xlabel='', ylabel='')
save_fig('04_coverage_watertype')

**Why these charts:** the map shows *where* problems concentrate (colour) and where monitoring is intense (size). The volume-vs-rate bubble chart separates "measured a lot" from "often bad". The site × month heat map exposes gaps in coverage, and the water-type table compares location types (the closest available proxy for the brief's *MonitoringLocationTypeName*).

**🔎 Inference (E4):** _written after the run_

## 5 · Outcome analysis (brief §5E) · answers **Q6** (exceedance / high-risk observations)
**What:** exceedances and high-risk results by parameter, shown as both **counts** and **rates**, and organizations compared by volume vs rate.
**Why:** the brief requires separating *high volume* from *high rate*. A parameter or organization can have the most exceedances simply because it measures the most. **No causal claims** are made here, only patterns.

In [ ]:
oc = q('''SELECT param, count(*) results, sum(exceedance_flag) exceedances, avg(exceedance_flag) * 100 exc_rate,
                 sum(high_risk_flag) high_risk, avg(high_risk_flag) * 100 hr_rate, count(DISTINCT site) FILTER (WHERE exceedance_flag = 1) sites_with_exc
          FROM a GROUP BY param''').set_index('param').loc[PARAMS].rename(index=SHORT)
display(save_tab(oc.round(3), '05_Q6_exceedance_by_parameter', index=True))
og = orgc[orgc.results >= 5000].copy()
og['exceedances'] = (og.exc_rate / 100 * og.results).round()
display(save_tab(og.sort_values('exceedances', ascending=False).head(10).round(2), '05_org_volume_vs_rate'))
fig, axes = plt.subplots(1, 3, figsize=(20, 4.8))
axes[0].bar(oc.index, oc.exceedances / 1e3, color='#DD8452'); axes[0].set(title='E5a · Exceedance COUNT (thousands)'); axes[0].tick_params(axis='x', rotation=20)
axes[1].bar(oc.index, oc.exc_rate, color='#C44E52'); axes[1].set(title='E5b · Exceedance RATE (% of results)'); axes[1].tick_params(axis='x', rotation=20)
for ax_, col in [(axes[0], 'exceedances'), (axes[1], 'exc_rate')]:
    for i, v_ in enumerate(oc[col]): ax_.text(i, (v_ / 1e3 if col == 'exceedances' else v_), f'{v_ / 1e3:.0f}K' if col == 'exceedances' else f'{v_:.1f}%', ha='center', va='bottom', fontsize=8)
axes[2].scatter(og.results / 1e3, og.exc_rate, s=20, alpha=0.6, color='#4C72B0'); axes[2].set_xscale('log')
for _, r_ in og.sort_values('exceedances', ascending=False).head(6).iterrows():
    axes[2].annotate(r_.org, (r_.results / 1e3, r_.exc_rate), fontsize=7.5, xytext=(3, 3), textcoords='offset points')
axes[2].set(title='E5c · Organizations (≥5K results): volume vs exceedance rate', xlabel='results (thousands, log)', ylabel='exceedance rate %')
save_fig('05_outcome_count_vs_rate')

**Why these charts:** the same exceedances shown twice, as a count and as a rate, make the volume-vs-rate difference obvious. The scatter places each organization by how much it measures and how often its results exceed thresholds.

**🔎 Inference (E5):** _written after the run_

## 6 · Relationships among parameters · answers **Q5**
**What:** correlation between parameters **measured at the same site on the same day** (co-located data, as the brief requires). For each site-day the median of each parameter is taken, and pairs are correlated.
- **Pearson** measures straight-line relationships.
- **Spearman** uses ranks, so it is robust to skew and outliers; it is preferred for turbidity, conductivity and nitrate.

**Why:** these relationships guide the regression model (Step 10). Physics says warmer water holds **less** oxygen, so a negative temperature–DO correlation is expected and confirms the data behaves sensibly.
**Caution:** correlation ≠ causation.

In [ ]:
sd = q('''PIVOT (SELECT site, dt, param, median(value_std) v FROM a GROUP BY ALL) ON param USING first(v) GROUP BY site, dt''')
sd = sd.rename(columns=SHORT); cols = [SHORT[p] for p in PARAMS]
print(f'Site-days: {len(sd):,}')
pairs = pd.DataFrame(index=cols, columns=cols, dtype=float)
for i in cols:
    for j in cols: pairs.loc[i, j] = (sd[i].notna() & sd[j].notna()).sum()
save_tab(pairs.astype(int), '06_Q5_pair_counts', index=True); display(pairs.astype(int))
pear = sd[cols].corr(method='pearson', min_periods=200); spear = sd[cols].corr(method='spearman', min_periods=200)
save_tab(pear.round(3), '06_Q5_pearson', index=True); save_tab(spear.round(3), '06_Q5_spearman', index=True)
fig, axes = plt.subplots(1, 3, figsize=(21, 5.6), gridspec_kw={'width_ratios': [1, 1, 1.05]})
mask = np.triu(np.ones_like(pear, dtype=bool), 1)
sns.heatmap(pear, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', vmin=-1, vmax=1, ax=axes[0]); axes[0].set(title='E6a · Pearson (site-day medians)')
sns.heatmap(spear, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', vmin=-1, vmax=1, ax=axes[1]); axes[1].set(title='E6b · Spearman (rank-based)')
td = sd[['Temperature', 'Dissolved oxygen']].dropna()
hb = axes[2].hexbin(td.Temperature, td['Dissolved oxygen'], gridsize=60, bins='log', cmap='viridis', extent=(0, 35, 0, 18))
plt.colorbar(hb, ax=axes[2], label='log10(site-days)')
tt = np.linspace(0, 35, 50); axes[2].plot(tt, 14.62 - 0.3898 * tt + 0.006969 * tt ** 2 - 0.00005897 * tt ** 3, color='red', lw=2, label='100% saturation (theory)')
axes[2].set(title=f'E6c · Temperature vs DO ({len(td):,} site-days)', xlabel='Temperature (°C)', ylabel='DO (mg/L)'); axes[2].legend()
save_fig('06_Q5_relationships')

**Why these charts:** two correlation heat maps (straight-line and rank-based) side by side show which relationships are robust. The hexbin plot shows the strongest pair in detail with a theoretical saturation curve, which checks the data against known physics.

**🔎 Inference (E6):** _written after the run_

## 7 · First answers to the mandatory business questions (EDA level)
Each answer is built from the tables above. The final answers come from the warehouse/OLAP (Step 9) and the models (Step 10).

In [ ]:
ans = pd.DataFrame([
 ('Q1', 'Water-quality measurements over time', 'E3a, E3e', f"Peak month {peak.month} ({peak.results:,}); lowest {low.month} ({low.results:,}); see E3e for parameter trends"),
 ('Q2', 'Compare parameters across sites', 'E04_Q2 table', 'Spread of site medians (P10–P90) per parameter'),
 ('Q3', 'Sites with extreme / poor quality', 'E04_Q3 tables, E4a', f"Worst sites listed per parameter (≥{MINN} results); % of poor sites per parameter"),
 ('Q4', 'Seasonal variation', 'E3f', 'Seasonal medians and exceedance rates per parameter'),
 ('Q5', 'Relationships among pH, DO, turbidity, nutrients', 'E6', f"Temperature–DO Spearman = {spear.loc['Temperature', 'Dissolved oxygen']:.2f}"),
 ('Q6', 'Exceedance / high-risk observations', 'E5', f"Highest exceedance rate: {oc.exc_rate.idxmax()} ({oc.exc_rate.max():.1f}%); most exceedances by count: {oc.exceedances.idxmax()}"),
 ('Q7', 'Site and organization sampling coverage', 'E4c, E04_Q7', f"Top 10 orgs supply {covq.loc['Share of results from top 10 orgs (%)', 'Value']}% of results"),
 ('Q8', 'Compare monitoring-location types', 'E4d', 'Surface vs ground vs ocean water (proxy until the Station file adds site types)'),
], columns=['Q', 'Question', 'Evidence', 'EDA answer'])
display(save_tab(ans, '07_question_answers'))

**🔎 Inference (E7):** _written after the run_

## Outputs
- `02_eda/figures/E*.png`: charts for the report and PPT
- `02_eda/tables/E*.csv`: evidence tables
- **Next:** Step 4, feature selection (`03_feature_selection/`)